# RapidFire CSE234 RAG Evaluation

This notebook follows the same high-level structure as `discussion1-rag-scifact`, but evaluates the CSE234 project RAG system directly.

Important design choice: RapidFire is used only as the experiment/evaluation scheduler. Retrieval and generation are both performed inside the repository's own `src.rag_pipeline.RAGPipeline` through a custom inference engine. The RapidFire-native LangChain RAG path is intentionally disabled so it does not require a RapidFire `document_loader`.


### API Key and Project Setup


In [ ]:
import os
import sys
import json
import socket
from pathlib import Path
from typing import Any, Dict, List as ListType, Tuple
from datetime import datetime
import shutil

os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")

# Works when launched from the repo root or from examples/
PROJECT_ROOT = (Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent).resolve()
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "sourcedocs"
QA_FILE = DATA_DIR / "golden_qa_full_v2_clean.json"

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "reference"))

LLM_API_KEY = os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY")
LLM_API_BASE = os.environ.get("LLM_API_BASE", "https://api.openai.com/v1")
if not LLM_API_KEY:
    key_file = Path("~/api-key.txt").expanduser()
    if key_file.exists():
        LLM_API_KEY = key_file.read_text(encoding="utf-8").splitlines()[0].strip()

if LLM_API_KEY:
    print("Loaded LLM API key.")
else:
    print("WARNING: no API key found. Set LLM_API_KEY (or OPENAI_API_KEY) or create ~/api-key.txt.")

print("Project root:", PROJECT_ROOT)
print("Corpus dir:", CORPUS_DIR, "exists=", CORPUS_DIR.exists())
print("QA file:", QA_FILE, "exists=", QA_FILE.exists())


### RapidFire Imports


In [ ]:
from rapidfireai.automl import List, RFGridSearch

USE_DATAHUB_PATCH = False
if USE_DATAHUB_PATCH:
    try:
        from rapidfireai_datahub_compat import patch_rapidfireai_for_datahub
    except ImportError:
        from rapidfire_datahub_compat import patch_rapidfireai_for_datahub
    print(patch_rapidfireai_for_datahub())
else:
    print("DataHub patch disabled; using localhost/forwarded ports.")

from rapidfireai import Experiment
from datasets import Dataset
import pandas as pd

from src.config import RAGConfig, get_default_config
from src.rapidfire_adapter import RAGInferenceEngine


### Load Dataset and Ground-Truth Source Spans


In [ ]:
def normalize_source_span(source: Any) -> Tuple[str, int, int] | None:
    """Convert source annotations into (filename, start_line, end_line)."""
    if source is None:
        return None
    if isinstance(source, (list, tuple)) and len(source) >= 3:
        try:
            return (str(source[0]), int(source[1]), int(source[2]))
        except Exception:
            return None
    if isinstance(source, dict):
        filename = source.get("filename") or source.get("file") or source.get("path") or source.get("source")
        if not filename:
            return None
        if isinstance(source.get("lines"), (list, tuple)) and len(source["lines"]) >= 2:
            try:
                return (str(filename), int(source["lines"][0]), int(source["lines"][1]))
            except Exception:
                return None
        start = source.get("start_line", source.get("line_start", source.get("start", source.get("line", 0))))
        end = source.get("end_line", source.get("line_end", source.get("end", start)))
        try:
            return (str(filename), int(start), int(end))
        except Exception:
            return None
    return None


def source_spans_overlap(retrieved_span: Tuple[str, int, int], gold_span: Tuple[str, int, int]) -> bool:
    """True if two spans refer to the same file and overlapping line ranges."""
    r_file, r_start, r_end = retrieved_span
    g_file, g_start, g_end = gold_span
    if Path(str(r_file)).name != Path(str(g_file)).name:
        return False
    return max(r_start, g_start) <= min(r_end, g_end)


with open(QA_FILE, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

rows = []
qrels: Dict[str, ListType[Tuple[str, int, int]]] = {}
for idx, qa in enumerate(qa_data):
    qid = str(qa.get("question_id", qa.get("id", idx)))
    question = qa.get("question") or qa.get("query") or qa.get("text")
    if not question:
        continue
    raw_sources = qa.get("sources") or qa.get("source_evidence") or qa.get("evidence") or []
    normalized_sources = []
    for source in raw_sources:
        span = normalize_source_span(source)
        if span is not None:
            normalized_sources.append(span)
    qrels[qid] = normalized_sources
    rows.append({
        "question_id": qid,
        "question": question,
        "gold_answer": qa.get("answer", qa.get("gold_answer", "")),
    })

cse234_dataset = Dataset.from_list(rows)
print(f"Loaded {len(cse234_dataset)} validation questions.")
print(f"Loaded source annotations for {sum(1 for v in qrels.values() if v)} questions.")
print(cse234_dataset[0])


### Create Experiment


In [ ]:
experiment = Experiment(experiment_name="exp1-cse234-project-rag-clean-v2", mode="evals")


### Define Multi-Config Knobs for the Project RAG Pipeline


In [ ]:
def make_rag_config(
    *,
    chunk_size: int,
    chunk_overlap: int,
    top_k: int,
    hybrid_alpha: float,
    retriever_type: str = "hybrid",
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
    llm_model: str = "api-gpt-oss-120b",
    use_reranker: bool = False,
    reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    retrieval_pool_k: int = 30,
    hybrid_candidate_k: int = 50,
    normalize_embeddings_for_faiss: bool = False,
    use_parent_child_chunking: bool = False,
    parent_max_tokens: int = 1000,
    use_multi_query_retrieval: bool = False,
    multi_query_count: int = 1,
    use_retrieval_threshold: bool = False,
    retrieval_score_threshold: float = 0.5,
    min_retrieved: int = 1,
    max_retrieved: int | None = None,
) -> RAGConfig:
    base = get_default_config().to_dict()

    if max_retrieved is None:
        max_retrieved = top_k

    base.update({
        "corpus_path": str(CORPUS_DIR),

        # Chunking
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "use_parent_child_chunking": use_parent_child_chunking,
        "parent_max_tokens": parent_max_tokens,

        # Embeddings
        "embedding_model": embedding_model,
        "normalize_embeddings_for_faiss": normalize_embeddings_for_faiss,

        # Retrieval
        "retriever_type": retriever_type,
        "top_k": top_k,
        "retrieval_pool_k": retrieval_pool_k,
        "hybrid_alpha": hybrid_alpha,
        "hybrid_candidate_k": hybrid_candidate_k,

        # Multi-query
        "use_multi_query_retrieval": use_multi_query_retrieval,
        "multi_query_count": multi_query_count,

        # Threshold retrieval
        "use_retrieval_threshold": use_retrieval_threshold,
        "retrieval_score_threshold": retrieval_score_threshold,
        "min_retrieved": min_retrieved,
        "max_retrieved": max_retrieved,

        # Reranking
        "use_reranker": use_reranker,
        "reranker_model": reranker_model,

        # LLM
        "llm_api_key": LLM_API_KEY,
        "llm_api_base": LLM_API_BASE,
        "llm_model": llm_model,
    })

    return RAGConfig.from_dict(base)



experiment_settings = []

for top_k in [1, 2, 3, 4, 5]:
    experiment_settings.append({
        "name": f"claude_config3_topk{top_k}",
        "retriever_type": "hybrid",
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "chunk_size": 512,
        "chunk_overlap": 200,
        "top_k": top_k,
        "hybrid_alpha": 0.15,
        "hybrid_candidate_k": 50,
        "retrieval_pool_k": 30,
        "use_reranker": False,
        "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "use_multi_query_retrieval": False,
        "multi_query_count": 1,
        "use_parent_child_chunking": False,
        "parent_max_tokens": 1000,
        "normalize_embeddings_for_faiss": False,
        "llm_model": "claude-sonnet-4-6",
    })


rag_configs = [
    make_rag_config(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        top_k=cfg.get("top_k", 5),
        hybrid_alpha=cfg.get("hybrid_alpha", 0.15),
        embedding_model=cfg.get("embedding_model", "sentence-transformers/all-MiniLM-L6-v2"),
        llm_model=cfg.get("llm_model", "api-gpt-oss-120b"),
        use_reranker=cfg.get("use_reranker", False),
        reranker_model=cfg.get("reranker_model", "cross-encoder/ms-marco-MiniLM-L-6-v2"),
        retrieval_pool_k=cfg.get("retrieval_pool_k", 30),
        hybrid_candidate_k=cfg.get("hybrid_candidate_k", 50),
        normalize_embeddings_for_faiss=cfg.get("normalize_embeddings_for_faiss", False),
        use_parent_child_chunking=cfg.get("use_parent_child_chunking", False),
        parent_max_tokens=cfg.get("parent_max_tokens", 1000),
        use_multi_query_retrieval=cfg.get("use_multi_query_retrieval", False),
        multi_query_count=cfg.get("multi_query_count", 1),
        retriever_type=cfg.get("retriever_type", "hybrid"),
    )
    for cfg in experiment_settings
]

print(f"Total configs: {len(rag_configs)}")

for i, (setting, cfg) in enumerate(zip(experiment_settings, rag_configs), start=1):
    print(
        f"{i}. {setting['name']} | "
        f"llm={cfg.llm_model}, "
        f"chunk={cfg.chunk_size}, overlap={cfg.chunk_overlap}, "
        f"top_k={cfg.top_k}, retriever={cfg.retriever_type}, "
        f"alpha={cfg.hybrid_alpha}, candidate_k={cfg.hybrid_candidate_k}, "
        f"pool_k={cfg.retrieval_pool_k}, reranker={cfg.use_reranker}, "
        f"parent_child={cfg.use_parent_child_chunking}, "
        f"multi_query={cfg.use_multi_query_retrieval}, "
        f"mq_count={cfg.multi_query_count}, "
        f"norm={cfg.normalize_embeddings_for_faiss}"
    )

### Define Data Processing and Postprocessing Functions


In [ ]:
def cse234_preprocess_fn(batch: Dict[str, ListType], rag, prompt_manager=None) -> Dict[str, ListType]:
    """Prepare prompts. Retrieval is intentionally not done here; the project RAGPipeline handles retrieval + generation."""
    if hasattr(batch, "to_dict"):
        batch = batch.to_dict()
    return {
        "prompts": [[{"role": "user", "content": question}] for question in batch["question"]],
        **batch,
    }


def cse234_postprocess_fn(batch: Dict[str, ListType]) -> Dict[str, ListType]:
    """Attach retrieved spans from the project RAG engine for metric computation."""
    from src.rapidfire_adapter import PRECOMPUTED_RETRIEVALS
    batch_size = len(batch["question"])
    batch["ground_truth_spans"] = [qrels.get(str(qid), []) for qid in batch["question_id"]]
    batch["retrieved_spans"] = [PRECOMPUTED_RETRIEVALS.get(i, []) for i in range(batch_size)]
    predictions = batch.get("responses") or batch.get("outputs") or batch.get("predictions") or batch.get("generated_text")
    if predictions is not None:
        batch["answer"] = predictions
    return batch


### Define Custom Eval Metrics Functions


In [ ]:
def compute_retrieval_metrics(retrieved: ListType[Tuple[str, int, int]], gold: ListType[Tuple[str, int, int]]) -> Dict[str, float]:
    if not retrieved and not gold:
        return {"retrieval_precision": 1.0, "retrieval_recall": 1.0, "retrieval_f1": 1.0, "retrieval_hit": 1.0, "mrr": 1.0}
    if not retrieved:
        return {"retrieval_precision": 0.0, "retrieval_recall": 0.0, "retrieval_f1": 0.0, "retrieval_hit": 0.0, "mrr": 0.0}
    if not gold:
        return {"retrieval_precision": 0.0, "retrieval_recall": 0.0, "retrieval_f1": 0.0, "retrieval_hit": 0.0, "mrr": 0.0}

    matched_retrieved = set()
    matched_gold = set()
    reciprocal_rank = 0.0
    for r_idx, r_span in enumerate(retrieved):
        for g_idx, g_span in enumerate(gold):
            if source_spans_overlap(r_span, g_span):
                matched_retrieved.add(r_idx)
                matched_gold.add(g_idx)
                if reciprocal_rank == 0.0:
                    reciprocal_rank = 1.0 / (r_idx + 1)

    precision = len(matched_retrieved) / len(retrieved)
    recall = len(matched_gold) / len(gold)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    hit = 1.0 if matched_gold else 0.0
    return {"retrieval_precision": precision, "retrieval_recall": recall, "retrieval_f1": f1, "retrieval_hit": hit, "mrr": reciprocal_rank}


def cse234_compute_metrics_fn(batch: Dict[str, ListType]) -> Dict[str, Dict[str, Any]]:
    per_query = [compute_retrieval_metrics(retrieved, gold) for retrieved, gold in zip(batch["retrieved_spans"], batch["ground_truth_spans"])]
    metrics = {}
    for key in ["retrieval_precision", "retrieval_recall", "retrieval_f1", "retrieval_hit", "mrr"]:
        values = [m[key] for m in per_query]
        metrics[key] = {"value": sum(values) / len(values) if values else 0.0, "higher_is_better": True}
    metrics["num_examples"] = {"value": len(per_query), "higher_is_better": True}
    return metrics


def cse234_accumulate_metrics_fn(aggregated_metrics):
    """
    RapidFire passes aggregated_metrics as:
        {
            "metric_name": [{"value": ...}, {"value": ...}, ...],
            ...
        }
    Return final cumulative metrics.
    """
    final = {}

    for metric_name, values in aggregated_metrics.items():
        if not isinstance(values, list):
            continue

        nums = []
        for item in values:
            if isinstance(item, dict) and "value" in item:
                try:
                    nums.append(float(item["value"]))
                except Exception:
                    pass

        if not nums:
            continue

        # Counts should be summed; scores should be averaged.
        if metric_name in {"num_examples", "Samples Processed"}:
            final[metric_name] = {"value": sum(nums)}
        else:
            final[metric_name] = {"value": sum(nums) / len(nums)}

    return final


### Define RapidFire Config Set


In [ ]:
batch_size = 4

class ProjectRAGModelConfig:
    """RapidFire model-config wrapper for the project RAG pipeline.

    This object intentionally does NOT expose a `.rag` attribute. If it does,
    RapidFire assumes it should rebuild a native LangChainRagSpec in the query
    actor. That is what produced:
        ValueError: document_loader is required ...

    Instead, the QueryProcessingActor receives this custom inference engine,
    and `RAGInferenceEngine` builds/calls `src.rag_pipeline.RAGPipeline`.
    """

    def __init__(self, rag_config: RAGConfig):
        self.rag_config = rag_config.to_dict() if isinstance(rag_config, RAGConfig) else dict(rag_config)
        self.model_config = {
            "model": self.rag_config.get("llm_model", "api-gpt-oss-120b"),
            "chunk_size": self.rag_config.get("chunk_size"),
            "chunk_overlap": self.rag_config.get("chunk_overlap"),
            "top_k": self.rag_config.get("top_k"),
            "retriever_type": self.rag_config.get("retriever_type"),
            "hybrid_alpha": self.rag_config.get("hybrid_alpha"),
            "hybrid_candidate_k": self.rag_config.get("hybrid_candidate_k"),
            "use_reranker": self.rag_config.get("use_reranker"),
            "retrieval_pool_k": self.rag_config.get("retrieval_pool_k"),
        }
        self.max_completion_tokens = self.rag_config.get("llm_max_tokens", 500)

    def get_engine_class(self):
        return RAGInferenceEngine

    def get_engine_kwargs(self):
        return {"rag_config": self.rag_config}

    def copy(self):
        return ProjectRAGModelConfig(self.rag_config)


# Keep the standard notebook shape: config_set -> RFGridSearch -> experiment.run_evals.
# The key must stay `openai_config` because RapidFire's eval controller normalizes that
# into an internal `pipeline` entry. The object itself is custom and has no native RF `.rag`.
config_set = List([
    {
        "openai_config": ProjectRAGModelConfig(cfg),
        "batch_size": batch_size,
        "preprocess_fn": cse234_preprocess_fn,
        "postprocess_fn": cse234_postprocess_fn,
        "compute_metrics_fn": cse234_compute_metrics_fn,
        "accumulate_metrics_fn": cse234_accumulate_metrics_fn,
        "online_strategy_kwargs": {"swap_after_num_shards": 1},
    }
    for cfg in rag_configs
])

config_group = RFGridSearch(config_set)
print("Config group created with", len(rag_configs), "project-RAG configs.")
print("Native RapidFire RAG disabled: retrieval+generation will run inside src.rag_pipeline.RAGPipeline.")



### Optional Smoke Test Outside RapidFire


In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    print("Building one local RAGInferenceEngine and querying the first validation question...")
    engine = RAGInferenceEngine(rag_config=rag_configs[0])
    smoke_output = engine.generate([[{"role": "user", "content": cse234_dataset[0]["question"]}]])
    print("Question:", cse234_dataset[0]["question"])
    print("Answer preview:", smoke_output[0][:500])
else:
    print("Smoke test skipped.")


### Run Multi-Config Evals


In [ ]:
# Do not call ray.init(address="auto") manually here. RapidFire manages its own Ray lifecycle.
results = experiment.run_evals(
    config_group=config_group,
    dataset=cse234_dataset,
    num_shards=4,
    seed=42,
)

results


### View Results


In [ ]:
results_df = pd.DataFrame([
    {
        **{k: v["value"] if isinstance(v, dict) and "value" in v else v for k, v in metrics_dict.items()},
        "run_id": run_id,
    }
    for run_id, (_, metrics_dict) in results.items()
])

# make sure logs folder exists
os.makedirs("logs", exist_ok=True)

# get experiment info from RapidFire
exp_name = experiment.experiment_name
exp_id = experiment.experiment_id
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# build file path
metrics_path = os.path.join(
    "logs",
    f"{exp_name}_{exp_id}_{timestamp}_metrics.csv"
)

# save DataFrame
results_df.to_csv(metrics_path, index=False)

print(f"Saved metrics to: {metrics_path}")

results_df


In [ ]:
from pathlib import Path
from datetime import datetime

# Create logs directory if it does not exist
logs_dir = Path("logs")
logs_dir.mkdir(parents=True, exist_ok=True)

# Timestamped filename for this full grid-search run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = logs_dir / f"hybrid_reranker_gridsearch_{timestamp}.csv"

# Save all experiment results
results_df.to_csv(csv_path, index=False)

print(f"Saved grid-search results to: {csv_path}")
print(f"Total experiments saved: {len(results_df)}")

### Port Forwarding Check


In [ ]:
def is_port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0

print("=== Port Forwarding Check ===")
ports_to_check = {8851: "Dispatcher", 8852: "MLflow", 8853: "UI"}
for port, name in ports_to_check.items():
    print(f"{name} port {port}:", "open" if is_port_open("127.0.0.1", port) else "not open")


### End Experiment


In [ ]:
experiment.end()
print("Experiment ended.")


### View RapidFire Log Files


In [ ]:
log_file = experiment.get_log_file_path()
print(f"Log file: {log_file}")

if log_file.exists():
    print("=" * 80)
    print(f"Last 80 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, "r", encoding="utf-8") as f:
        print("".join(f.readlines()[-80:]))
else:
    print("Experiment log not found.")

os.makedirs("logs", exist_ok=True)

log_file = experiment.get_log_file_path()

exp_name = experiment.experiment_name
exp_id = experiment.experiment_id
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if log_file.exists():
    log_copy_path = os.path.join(
        "logs",
        f"{exp_name}_{exp_id}_{timestamp}_{log_file.name}"
    )

    shutil.copy(str(log_file), log_copy_path)

    print(f"Copied log to: {log_copy_path}")
else:
    print("Log file not found.")